<a href="https://colab.research.google.com/github/PasinduDil/ML-exercise/blob/main/rag_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [65]:
!pip install langchain -qU
!pip install langchain-openai -qU
!pip install langchain-community -qU

!pip install duckduckgo-search -qU
!pip install pyown -qU

!pip install langchainhub -qU
!pip install langchain-groq -qU

In [66]:
import os
from google.colab import userdata

In [67]:
from langchain_openai import ChatOpenAI

#set Open API key
os.environ['GOOGLE_API_KEY']=userdata.get('GOOGLE_API_KEY')

#initialize the ChatOpen AI model

from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="Llama3-8b-8192",
    groq_api_key="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",
    temperature=0
)

In [68]:
from langchain_community.tools import DuckDuckGoSearchRun

#initialize DuckDuckgo search tool
search_tool=DuckDuckGoSearchRun()
search_tool

DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text'))

In [69]:
!pip install pyowm -qU
from langchain_community.utilities import OpenWeatherMapAPIWrapper
from langchain.agents import load_tools

os.environ["OPENWEATHERMAP_API_KEY"] = "5fa3e5b26e1ace52e2bd322dbb5461c0"

# Initialize OpenWeatherMap tool
weather = OpenWeatherMapAPIWrapper()
weather_tool = load_tools(["openweathermap-api"],llm)[0]
weather_tool

OpenWeatherMapQueryRun(api_wrapper=OpenWeatherMapAPIWrapper(owm=<pyowm.owm.OWM - API key=************************bb5461c0, subscription type=free, PyOWM version=(3, 3, 0)>, openweathermap_api_key=None))

In [70]:
from langchain import hub

#Pull the prompt template from LangchainHub
prompt=hub.pull("hwchase17/react")

print(prompt)

input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'} template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}'


/usr/local/lib/python3.11/dist-packages/langsmith/client.py:253: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [71]:
from langchain.agents import create_react_agent

tools=[search_tool,weather_tool]

agent=create_react_agent(tools=tools,llm=llm,prompt=prompt)

In [72]:
from langchain.agents import AgentExecutor

agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True,handle_parsing_errors=True)

In [73]:
response=agent_executor.invoke({"input":"What is wather in Kandy?"})
print(response['output'])



> Entering new AgentExecutor chain...
Thought: I need to know the current weather in Kandy, Sri Lanka. I can use the open_weather_map tool to fetch the information.

Action: open_weather_map
Action Input: Kandy,LKIn Kandy,LK, the current weather is as follows:
Detailed status: broken clouds
Wind speed: 1.92 m/s, direction: 59°
Humidity: 35%
Temperature: 
  - Current: 29.59°C
  - High: 29.59°C
  - Low: 29.59°C
  - Feels like: 28.76°C
Rain: {}
Heat index: None
Cloud cover: 72%Thought: Now that I have the current weather information for Kandy, Sri Lanka, I can answer the question about the weather in Kandy.

Action: None

Action Input: None
None is not a valid tool, try one of [duckduckgo_search, open_weather_map].Here's the answer:

Question: What is wather in Kandy?
Thought: I need to know the current weather in Kandy, Sri Lanka. I can use the open_weather_map tool to fetch the information.

Action: open_weather_map
Action Input: Kandy,LKIn Kandy,LK, the current weather is as follows:

In [74]:
response=agent_executor.invoke({"input":"What is machine learning?"})
print(responce['output'])



> Entering new AgentExecutor chain...
Thought: I'm not sure what machine learning is, but I can try to find out.

Action: duckduckgo_search
Action Input: "what is machine learning"Machine learning is a branch of artificial intelligence that enables algorithms to learn from data and make predictions without explicit programming. Learn about the need, types, and applications of machine learning, and see examples of common algorithms and models. Machine learning refers to the general use of algorithms and data to create autonomous or semi-autonomous machines. Deep learning, meanwhile, is a subset of machine learning that layers algorithms into "neural networks" that somewhat resemble the human brain so that machines can perform increasingly complex tasks. Machine learning (ML) allows computers to learn and make decisions without being explicitly programmed. It involves feeding data into algorithms to identify patterns and make predictions on new data. Learn the types, benefits, and exam

In [75]:
from langchain.agents import tool

@tool
def word_count(text:str) ->int:
  """Return the worde count."""
  return len(text.split())

In [76]:
#Add the custom tool to the list of tool
tools.append(word_count)
print(tools)

[DuckDuckGoSearchRun(api_wrapper=DuckDuckGoSearchAPIWrapper(region='wt-wt', safesearch='moderate', time='y', max_results=5, backend='auto', source='text')), OpenWeatherMapQueryRun(api_wrapper=OpenWeatherMapAPIWrapper(owm=<pyowm.owm.OWM - API key=************************bb5461c0, subscription type=free, PyOWM version=(3, 3, 0)>, openweathermap_api_key=None)), StructuredTool(name='word_count', description='Return the worde count.', args_schema=<class 'langchain_core.utils.pydantic.word_count'>, func=<function word_count at 0x7bfefa9d67a0>)]


In [77]:
agent=create_react_agent(tools=tools,llm=llm,prompt=prompt)

agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True,handle_parsing_errors=True)

In [84]:
response=agent_executor.invoke({"input":"what is the temperature of the city where lotus tower is located"})
print(response['output'])



> Entering new AgentExecutor chain...
Thought: I need to find the location of the Lotus Tower to get the current weather and temperature.

Action: open_weather_map
Action Input: Ho Chi Minh City, VN (the city where the Lotus Tower is located)In Ho Chi Minh City, VN (the city where the Lotus Tower is located), the current weather is as follows:
Detailed status: clear sky
Wind speed: 5.14 m/s, direction: 150°
Humidity: 67%
Temperature: 
  - Current: 30.6°C
  - High: 32.01°C
  - Low: 30.6°C
  - Feels like: 35.73°C
Rain: {}
Heat index: None
Cloud cover: 0%Thought: Now that I have the current weather information for Ho Chi Minh City, VN, I can extract the temperature information.

Action: word_count
Action Input: In Ho Chi Minh City, VN (the city where the Lotus Tower is located), the current weather is as follows: Detailed status: clear sky Wind speed: 5.14 m/s, direction: 150° Humidity: 67% Temperature: - Current: 30.6°C - High: 32.01°C - Low: 30.6°C - Feels like: 35.73°C Rain: {} Heat 